# Lot A2 — MARBERTv2 en cascade : validation croisée 5 plis × 5 graines

Protocole : `docs/lot-a2_modele_tn_protocole.md` (§3, avenant 1 et note d'exécution).

- **Données** : `kaggle_train.tsv` (jeu de données **privé**) : 800 commentaires TSAC en lettres arabes,
  colonnes `label` (1 = tunisien, 0 = arabe standard), `lex` (1 = marque du lexique trouvée), `text`.
- **Le test gelé n'est ni chargé ni lu ici.** Ce notebook ne fait que régler le modèle sur l'entraînement.
- **Cascade** : marque du lexique → tunisien ; sinon, tunisien si la probabilité du modèle atteint le seuil.
- **Réglage** : taux d'apprentissage et nombre d'époques choisis sur les prédictions hors pli ;
  seuil = meilleur rappel tunisien avec une précision d'au moins 0,90.
- **Sortie** : des scores seulement (aucun texte), dans `/kaggle/working/marbert_cv_results.json`.

**Réglages du notebook (panneau de droite)** : Accelerator = **GPU T4**, Internet = **On**,
Input = ton jeu de données privé contenant `kaggle_train.tsv`.

## 1. Vérifications : GPU et fichier de données

In [2]:
# Le GPU doit apparaître ; sinon : Settings > Accelerator > GPU T4
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Le fichier d'entraînement doit être trouvé une seule fois
import glob
print(glob.glob("/kaggle/input/**/kaggle_train.tsv", recursive=True))

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
['/kaggle/input/datasets/souriabdelmouhib/newscheck-a2-tn-train800/kaggle_train.tsv']


## 2. Bibliothèques

In [3]:
# Toutes déjà installées sur Kaggle (torch, transformers, scikit-learn, pandas)
import gc
import glob
import json
import os
import random
import statistics
import time

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

## 3. Configuration (fixée avant le lancement)

In [4]:
# --- Configuration, fixée avant le lancement ---------------------------------------------
MODEL = os.environ.get("A2_MODEL", "UBC-NLP/MARBERTv2")  # A2_MODEL : seulement pour un test local
SEEDS = [int(s) for s in os.environ.get("A2_SEEDS", "1,2,3,4,5").split(",")]
FOLDS = 5
LEARNING_RATES = [float(x) for x in os.environ.get("A2_LRS", "2e-5,5e-5").split(",")]
EPOCHS = int(os.environ.get("A2_EPOCHS", "4"))  # chaque époque de 1 à EPOCHS est évaluée
BATCH = 16
MAX_LEN = 128
WARMUP = 0.1
MIN_PRECISION = 0.90  # protocole §3 : précision tn d'au moins 0,90
THRESHOLDS = [round(x, 2) for x in np.arange(0.05, 1.0, 0.05)] + [0.97, 0.99]
OUT = os.environ.get("A2_OUT", "/kaggle/working/marbert_cv_results.json")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 4. Fonctions : métriques, cascade, choix du seuil, entraînement d'un pli

In [5]:
def find_data() -> str:
    """Chemin de kaggle_train.tsv dans les jeux de données ajoutés au notebook."""
    paths = glob.glob(os.environ.get("A2_DATA", "/kaggle/input/**/kaggle_train.tsv"), recursive=True)
    if len(paths) != 1:
        raise SystemExit(f"kaggle_train.tsv introuvable ou en double : {paths}")
    return paths[0]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def scores(labels, preds) -> dict:
    """Rappel, précision tn et macro-F1 tn / ar."""
    labels, preds = np.asarray(labels), np.asarray(preds)
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    recall = tp / (tp + fn) if tp + fn else 0.0
    precision = tp / (tp + fp) if tp + fp else 1.0  # aucune prédiction tn : aucune erreur tn
    f1_tn = 2 * tp / (2 * tp + fp + fn) if tp else 0.0
    f1_ar = 2 * tn / (2 * tn + fn + fp) if tn else 0.0
    return {"recall": recall, "precision": precision, "macro_f1": (f1_tn + f1_ar) / 2}


def cascade(lex, probs, threshold) -> np.ndarray:
    """Marque du lexique → tn ; sinon tn si la probabilité atteint le seuil."""
    return ((np.asarray(lex) == 1) | (np.asarray(probs) >= threshold)).astype(int)


def choose_threshold(labels, lex, probs) -> float:
    """Seuil au meilleur rappel tn avec précision ≥ 0,90 ; à égalité, le plus haut (prudent)."""
    best = None
    for t in THRESHOLDS:
        s = scores(labels, cascade(lex, probs, t))
        if s["precision"] >= MIN_PRECISION and (best is None or s["recall"] >= best[1]):
            best = (t, s["recall"])
    return best[0] if best else THRESHOLDS[-1]  # repli : le seuil le plus strict


def predict(model, loader) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            logits = model(**batch).logits
            out.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def make_loader(tokenizer, texts, labels, shuffle, seed):
    enc = tokenizer(list(texts), truncation=True, max_length=MAX_LEN, padding=True,
                    return_tensors="pt")
    items = [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i],
              "labels": torch.tensor(int(labels[i]))} for i in range(len(texts))]
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(items, batch_size=BATCH, shuffle=shuffle, generator=generator)


def train_fold(tokenizer, texts, labels, train_idx, val_idx, lr, seed) -> list[np.ndarray]:
    """Affine le modèle sur un pli ; renvoie les probabilités du pli de validation après chaque époque."""
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(DEVICE)
    train = make_loader(tokenizer, texts[train_idx], labels[train_idx], True, seed)
    val = make_loader(tokenizer, texts[val_idx], labels[val_idx], False, seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = EPOCHS * len(train)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    per_epoch = []
    for _ in range(EPOCHS):
        model.train()
        for batch in train:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                loss = model(**batch).loss
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        per_epoch.append(predict(model, val))
    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()
    return per_epoch

## 5. Validation croisée complète

Durée indicative sur GPU T4 : de l'ordre de 30 à 60 minutes
(2 taux d'apprentissage × 5 graines × 5 plis = 50 entraînements de 4 époques).

In [ ]:
def main() -> None:
    start = time.time()
    # quoting=3 (QUOTE_NONE) : un guillemet " dans un commentaire est un caractère ordinaire ;
    # sans cela, pandas fusionne plusieurs lignes (bug de la première exécution : 625 au lieu de 800)
    path = find_data()
    data = pd.read_csv(path, sep="\t", quoting=3, dtype={"text": str}, keep_default_na=False)
    with open(path, encoding="utf-8") as handle:
        expected = sum(1 for _ in handle) - 1  # lignes du fichier, sans l'en-tête
    if len(data) != expected or not set(data["label"]) <= {0, 1} or not set(data["lex"]) <= {0, 1}:
        raise SystemExit(f"lecture incorrecte : {len(data)} lignes lues pour {expected} dans le fichier")
    texts, labels, lex = data["text"].to_numpy(), data["label"].to_numpy(), data["lex"].to_numpy()
    print(f"{len(data)} textes (tn {labels.sum()}, ar {len(labels) - labels.sum()}) ; {DEVICE}")
    base = scores(labels, lex)
    print(f"baseline lexique seul : rappel {base['recall']:.3f}, précision {base['precision']:.3f},"
          f" macro-F1 {base['macro_f1']:.3f}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL)

    # oof[(lr, epoch)][seed] = probabilités hors pli de chaque texte
    oof = {(lr, e): {} for lr in LEARNING_RATES for e in range(1, EPOCHS + 1)}
    for lr in LEARNING_RATES:
        for seed in SEEDS:
            probs = {e: np.zeros(len(data)) for e in range(1, EPOCHS + 1)}
            folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
            for k, (tr, va) in enumerate(folds.split(texts, labels), start=1):
                for e, p in enumerate(train_fold(tokenizer, texts, labels, tr, va, lr, seed), 1):
                    probs[e][va] = p
                print(f"  lr {lr:g} · graine {seed} · pli {k}/{FOLDS} ({time.time() - start:.0f} s)")
            for e, values in probs.items():
                oof[(lr, e)][seed] = values

    # Choix sur l'entraînement seulement : seuil commun aux graines, puis scores par graine
    results = []
    for (lr, e), by_seed in oof.items():
        pooled = np.concatenate([by_seed[s] for s in SEEDS])
        threshold = choose_threshold(np.tile(labels, len(SEEDS)), np.tile(lex, len(SEEDS)), pooled)
        per_seed = [scores(labels, cascade(lex, by_seed[s], threshold)) for s in SEEDS]
        row = {"lr": lr, "epochs": e, "threshold": threshold}
        for key in ("recall", "precision", "macro_f1"):
            values = [s[key] for s in per_seed]
            row[key] = statistics.mean(values)
            row[key + "_sd"] = statistics.stdev(values) if len(values) > 1 else 0.0
        results.append(row)
        print(f"lr {lr:g} · {e} époque(s) · seuil {threshold} : rappel {row['recall']:.3f} ±"
              f" {row['recall_sd']:.3f} ; précision {row['precision']:.3f} ±"
              f" {row['precision_sd']:.3f} ; macro-F1 {row['macro_f1']:.3f} ± {row['macro_f1_sd']:.3f}")
    # Règle fixée avant : meilleur rappel moyen parmi les configurations à précision moyenne ≥ 0,90 ;
    # égalité : taux d'apprentissage le plus petit, puis le moins d'époques.
    eligible = [r for r in results if r["precision"] >= MIN_PRECISION] or results
    chosen = max(eligible, key=lambda r: (round(r["recall"], 3), -r["lr"], -r["epochs"]))
    print(f"\nretenu : lr {chosen['lr']:g}, {chosen['epochs']} époque(s), seuil {chosen['threshold']}"
          f" ; rappel {chosen['recall']:.3f} ± {chosen['recall_sd']:.3f}, précision"
          f" {chosen['precision']:.3f} ± {chosen['precision_sd']:.3f}")
    print("Préliminaire et optimiste : seuil et réglages choisis sur ces mêmes prédictions hors pli."
          " Le test gelé (300 textes, double annotation) fera foi.")
    with open(OUT, "w", encoding="utf-8") as handle:
        json.dump({"model": MODEL, "seeds": SEEDS, "folds": FOLDS, "n": len(data),
                   "baseline_lexique": base, "grid": results, "chosen": chosen,
                   "minutes": round((time.time() - start) / 60, 1)}, handle, indent=1)
    print(f"scores écrits : {OUT}")


main()

## 6. Résumé des scores (à copier pour le journal EXP-A2-015)

In [10]:
import json

# Relire le fichier de résultats écrit par la cellule 5 (des scores seulement, aucun texte)
with open(OUT, encoding="utf-8") as handle:
    results = json.load(handle)

# Résumé : modèle, nombre de textes (doit être 800), baseline du lexique seul, configuration retenue, durée
print(json.dumps({k: results[k] for k in ("model", "n", "baseline_lexique", "chosen", "minutes")},
                 indent=1, ensure_ascii=False))

# Grille complète : une ligne par (taux d'apprentissage, nombre d'époques)
grid = pd.DataFrame(results["grid"])
grid["lr"] = grid["lr"].map(lambda x: f"{x:g}")  # affiche 2e-05 / 5e-05 au lieu de 0.0
grid.round(3)

{
 "model": "UBC-NLP/MARBERTv2",
 "n": 800,
 "baseline_lexique": {
  "recall": 0.48012718600953896,
  "precision": 0.967948717948718,
  "macro_f1": 0.5652447352846792
 },
 "chosen": {
  "lr": 2e-05,
  "epochs": 4,
  "threshold": 0.3,
  "recall": 0.9659777424483307,
  "recall_sd": 0.004976939854530544,
  "precision": 0.9006950142120768,
  "precision_sd": 0.0032419608610312497,
  "macro_f1": 0.8169537326232564,
  "macro_f1_sd": 0.006354349423767751
 },
 "minutes": 23.7
}


,lr,epochs,threshold,recall,recall_sd,precision,precision_sd,macro_f1,macro_f1_sd
0,2e-05,1,0.80,0.838,0.033,0.913,0.008,0.743,0.019
1,2e-05,2,0.55,0.948,0.022,0.904,0.010,0.807,0.008
2,2e-05,3,0.35,0.963,0.007,0.905,0.006,0.823,0.007
3,2e-05,4,0.30,0.966,0.005,0.901,0.003,0.817,0.006
4,5e-05,1,0.60,0.918,0.032,0.908,0.016,0.790,0.011
5,5e-05,2,0.40,0.960,0.008,0.904,0.011,0.817,0.021
6,5e-05,3,0.20,0.958,0.008,0.903,0.007,0.815,0.008
7,5e-05,4,0.15,0.962,0.005,0.907,0.005,0.825,0.009


**Rappel** : ces chiffres sont **préliminaires et optimistes** (seuil et réglages choisis sur les mêmes
prédictions hors pli). Seul le test gelé de 300 textes, annoté par deux personnes puis adjugé,
fera foi — mesuré une seule fois, avec la configuration retenue ici.